In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Titanic") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()


In [0]:
df = spark.read.format("csv") \
               .option("header", "true") \
               .option("inferSchema", "true") \
               .load("/Volumes/workspace/default/files/train.csv")

In [0]:
df.limit(5).display()

## Explaining a select

To see the plan being built we use `explain`.  `explain` is not a transformation in that it doesn't
create a new DataFrame, and it isn't an action because it doesn't trigger execution of the plan.  It is a debugging 
tool.



In [0]:
df.select("Age").explain(extended=True)

Now let's considering the parts of the plan above.

### The Parsed Logical Plan

```

== Parsed Logical Plan ==
'Project ['Age]
+- Relation [PassengerId#11080,Survived#11081,Pclass#11082,Name#11083,Sex#11084,Age#11085,SibSp#11086,Parch#11087,Ticket#11088,Fare#11089,Cabin#11090,Embarked#11091] csv
```

Parsing turns your code or SQL into an abstract syntax tree.  This part of the plan is about syntax.

`Project` = projection = reduce the dimensionality of the data by reducing the number of columns or transforming columns.  It is  the low-level operation that implements `select`.

`Relation` is Spark's logical-plan name for the source dataset.

Plans are read from the bottom-up.   The relation is fed into the projection.

The `+-` can be thought of as an arrow in this case representing the direction data flows up the tree from the relation to the projection.

### The Analyzed Logical Plan

```

== Analyzed Logical Plan ==
Age: double
Project [Age#11085]
+- Relation [PassengerId#11080,Survived#11081,Pclass#11082,Name#11083,Sex#11084,Age#11085,SibSp#11086,Parch#11087,Ticket#11088,Fare#11089,Cabin#11090,Embarked#11091] csv
```

The Analyzed Logical Plan focuses on semantics.  Spark has resolved `Age` to a specific column `Age#11085` where `#11085` is an internal identifier.  It has determined that `Age` has type `double.`  

Spark has also validated that the column exists and that the query is well-defined.

### The Optimized Logical Plan

```

== Optimized Logical Plan ==
Project [Age#11085]
+- Relation [PassengerId#11080,Survived#11081,Pclass#11082,Name#11083,Sex#11084,Age#11085,SibSp#11086,Parch#11087,Ticket#11088,Fare#11089,Cabin#11090,Embarked#11091] csv
```

Spark at this point would have rearranged the order of transformations if it
would likely improve performance, such as moving filters earlier to avoid
performing operations on data that will later be omitted.  

In this case we are only performibng one transformation, so there is little to optimize.
The logical plan therefore appears unchanged, although some
optimizations (such as column pruning) will be reflected in the physical plan. 

### The Physical Plan

```

== Physical Plan ==
FileScan csv [Age#11085] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[dbfs:/Volumes/workspace/default/files/train.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<Age:double>
```

The physical plan describes how Spark will actually execute the query on the cluster.

In this case Spark will perform a `FileScan`, meaning it will read data from a file rather than from an in-memory table or from the output of some earlier computation.

The file format is `csv`, and the file location is `dbfs:/Volumes/workspace/default/files/train.csv`.

The scan is only reading the column `Age`, shown both by `[Age#11085]` and by `ReadSchema: struct<Age:double>`.  This is an example of **column pruning**: even though the CSV file contains many columns, Spark will only read the one needed by the query.

`DataFilters: []` means there are no row filters such as `Age > 30`.

`PartitionFilters: []` means there are no filters on partition columns. This matters more when data is stored in partitioned directories.

`PushedFilters: []` means no filters were pushed down into the file scan itself. If there had been a filter, Spark might have been able to apply part of it while reading the data.

`Batched: false` means Spark is not using a batched or vectorized reader here. CSV generally does not support the same efficient batched reading that columnar formats like Parquet do.

So the physical plan is saying: read the `Age` column as a `double` from this CSV file, with no filtering and no partition pruning.


## Something More Complex

```df.select("age", when(col("age") < 12, "kid")).limit(10)```

As in ordinary Python, the inner function calls are evaluated first. However, in Spark this does not execute the query. Instead, Python constructs Column expressions and DataFrame transformations, which together build a logical plan for later execution.

`col("age")` creates a `Column` expression referring to a column `age`, but at this point it is just a symbolic reference because`"age"` is not yet resolved.

`col("age") < 12` creates a boolean `Column` expression that evaluates to true when data in the unresolved referenced column "age" is less than 12.  

`when` creates a conditional column expression.  It means: if the boolean expression is true for a row, place "kid" in the output column. If no .otherwise(...) is provided, then null is produced when the condition is false.

The `df.select("age", ...)` selects a column "age" from Dataframe `df`.  

`df.select` returns a DataFrame upon which we call `limit(10)` to limit the result to at most 10 rows.  Thus, this could've been written

```
  boolean_exp = col("age") < 12
  exp = when(boolean_exp, "kid")
  df2 = df.select(["age", exp]
  df3 = df2.limit(10)
```

where `boolean_exp` refers to a boolean expression. 



In [0]:
from pyspark.sql.functions import col, when

display(df.select("age", when(col("age") < 12, "kid")).limit(10))


### Explaining Something More Complex

In [0]:
df.select(["age", when(col("age") < 12, "kid")]).explain(extended=True)


```

== Parsed Logical Plan ==
'Project ['age, unresolvedalias('when('`<`('age, 12), kid))]
+- Relation [PassengerId#11080,Survived#11081,Pclass#11082,Name#11083,Sex#11084,Age#11085,SibSp#11086,Parch#11087,Ticket#11088,Fare#11089,Cabin#11090,Embarked#11091] csv

== Analyzed Logical Plan ==
age: double, CASE WHEN (age < 12) THEN kid END: string
Project [age#11085, CASE WHEN (age#11085 < cast(12 as double)) THEN kid END AS CASE WHEN (age < 12) THEN kid END#11130]
+- Relation [PassengerId#11080,Survived#11081,Pclass#11082,Name#11083,Sex#11084,Age#11085,SibSp#11086,Parch#11087,Ticket#11088,Fare#11089,Cabin#11090,Embarked#11091] csv

== Optimized Logical Plan ==
Project [age#11085, CASE WHEN (age#11085 < 12.0) THEN kid END AS CASE WHEN (age < 12) THEN kid END#11130]
+- Relation [PassengerId#11080,Survived#11081,Pclass#11082,Name#11083,Sex#11084,Age#11085,SibSp#11086,Parch#11087,Ticket#11088,Fare#11089,Cabin#11090,Embarked#11091] csv

== Physical Plan ==
PhotonResultStage
+- PhotonColumnarToRow
   +- PhotonProject [age#11085, CASE WHEN (age#11085 < 12.0) THEN kid END AS CASE WHEN (age < 12) THEN kid END#11130]
      +- PhotonRowToColumnar
         +- FileScan csv [Age#11085] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[dbfs:/Volumes/workspace/default/files/train.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<Age:double>

```


Hmmm it seems that the free edition is not like the deprecated community edition.  It does not support access to parallelize.

In [0]:
rdd = spark.sparkContext.parallelize([1, 2, 3, 4])
rdd2 = rdd.map(lambda x: x * 2)  # Narrow transformation
print(rdd2.collect())  # Output: [2, 4, 6, 8]
